# FloodNav — Flood-Aware A* Route Optimizer

**Stack:** OSMnx · NetworkX A* · thaiwater.net · TMD live data  
**Area:** Chiang Rai province road network (เชียงราย)  
**Goal:** Find lowest-flood-risk path, not just shortest path

---
```
Phase 1 (this notebook): A* with flood-weighted edges
Phase 2 (future):        GAT (Graph Attention Network) replaces hand-tuned weights
```


In [ ]:
%pip install osmnx networkx geopandas folium requests shapely numpy pandas matplotlib --quiet

In [ ]:
import osmnx as ox
import networkx as nx
import numpy as np
import pandas as pd
import folium
import requests
import json
import math
import pickle
import os
import pathlib
from IPython.display import display, HTML

ox.settings.log_console = False

print('Libraries loaded ✓')

## 1 · Config

In [ ]:
# Project root: works whether Jupyter is launched from scripts/ or project root
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'scripts' else pathlib.Path.cwd()
print('ROOT:', ROOT)

FLOODNAV_BASE = 'http://localhost:3001/api'

# River monitoring stations in Chiang Rai (thaiwater.net)
RIVER_STATIONS = [
    {'id': 'G.2A',  'name': 'แม่น้ำกก บ้านกกโท้ง',   'lat': 19.921, 'lon': 99.849},
    {'id': 'G.7',   'name': 'แม่น้ำกก สะพานสบกก',     'lat': 20.228, 'lon': 99.882},
    {'id': 'Kh.89', 'name': 'แม่น้ำจัน บ้านหัวสะพาน', 'lat': 20.158, 'lon': 99.843},
    {'id': 'I.14',  'name': 'แม่น้ำอิง บ้านน้ำอิง',   'lat': 19.833, 'lon': 100.088},
]

# Match FloodNav route endpoints exactly — Chiang Rai province
ROUTE_ENDPOINTS = {
    'A': {'origin': (19.908, 99.832), 'dest': (20.434, 99.882), 'name': 'ทล.1 เมือง→แม่สาย',         'color': '#22c55e'},
    'B': {'origin': (19.908, 99.832), 'dest': (19.977, 100.074),'name': 'ทล.1020 เมือง→เทิง',        'color': '#f59e0b'},
    'C': {'origin': (19.908, 99.832), 'dest': (19.375, 99.858), 'name': 'ทล.118 เมือง→เวียงป่าเป้า', 'color': '#ef4444'},
}

# Flood-weight tuning parameters
RIVER_DANGER_RADIUS_M = 3000
FLOOD_PENALTY_MAX     = 9.0
FLOOD_MARGIN_M        = 5.0   # margin (m) below warning level = full safety

# File paths — all relative to project ROOT
GRAPH_CACHE = str(ROOT / 'chiang_rai_graph.pkl')
OSM_FILE    = str(ROOT / 'chiang_rai.osm')
PBF_FILE    = str(ROOT / 'chiang_rai.osm.pbf')

print('Config OK ✓')

## 2 · Pull Live Data from FloodNav API

In [ ]:
def fetch_live_data():
    live = {'water_levels': [], 'dam_levels': [], 'route_risk': {}, 'rain_mm': 0.0}
    endpoints = {
        'water_levels': f'{FLOODNAV_BASE}/water-levels',
        'dam_levels':   f'{FLOODNAV_BASE}/dams',
        'route_risk':   f'{FLOODNAV_BASE}/route-risk',
    }
    for key, url in endpoints.items():
        try:
            r = requests.get(url, timeout=8)
            if r.ok:
                live[key] = r.json()
                print(f'  ✓ {key}')
            else:
                print(f'  ✗ {key}: HTTP {r.status_code}')
        except Exception as e:
            print(f'  ✗ {key}: {e}')
    return live

print('Fetching live data from FloodNav server...')
live_data = fetch_live_data()

# Summarise signals
wl = live_data['water_levels']
dm = live_data['dam_levels']
rr = live_data['route_risk']

online_stations = [s for s in wl if s.get('level') and s.get('warning_level')]
# margin-based: 0 = safe (>5m below warning), 1 = at/above warning level
water_ratios = [
    max(0.0, min(1.0, 1 - (s['warning_level'] - s['level']) / FLOOD_MARGIN_M))
    for s in online_stations
]
WATER_PRESSURE = np.mean(water_ratios) if water_ratios else 0.3

online_dams  = [d for d in dm if d.get('percent') is not None]
DAM_PRESSURE = np.mean([min(d['percent'] / 100, 1.1) for d in online_dams]) if online_dams else 0.5

print(f'\nLive signals:')
print(f'  Water pressure (margin-based): {WATER_PRESSURE:.3f}')
print(f'  Dam pressure (% capacity):     {DAM_PRESSURE:.3f}')
if rr:
    for rid, v in rr.items():
        print(f'  ML risk Route {rid}: {v["risk"]}% (confidence {v["confidence"]}%)')

## 3 · Load Road Network

In [ ]:
def load_graph():
    # 1) Pickle cache (fastest)
    if os.path.exists(GRAPH_CACHE):
        print(f'Loading road network from cache ({GRAPH_CACHE})...')
        with open(GRAPH_CACHE, 'rb') as f:
            G = pickle.load(f)
        print(f'Loaded: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
        return G

    # 2) Local OSM XML (converted from PBF via: osmium cat chiang_rai.osm.pbf -o chiang_rai.osm)
    if os.path.exists(OSM_FILE):
        print(f'Building graph from local OSM XML ({OSM_FILE})...')
        G = ox.graph_from_xml(OSM_FILE, retain_all=False, simplify=True)
    else:
        # 3) Download from Overpass API — osmnx 2.x: bbox=(left, bottom, right, top)
        print('Downloading road network from OSM (~60–120 s for Chiang Rai bbox)...')
        G = ox.graph_from_bbox(
            bbox=(99.35, 19.20, 100.55, 20.55),  # (west, south, east, north)
            network_type='drive',
            simplify=True,
            retain_all=False,
        )

    with open(GRAPH_CACHE, 'wb') as f:
        pickle.dump(G, f)
    print(f'Cached → {GRAPH_CACHE}')
    print(f'{G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges')
    return G

G = load_graph()

## 4 · Flood Feature Engineering

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6_371_000
    φ1, φ2 = math.radians(lat1), math.radians(lat2)
    dφ = math.radians(lat2 - lat1)
    dλ = math.radians(lon2 - lon1)
    a  = math.sin(dφ/2)**2 + math.cos(φ1)*math.cos(φ2)*math.sin(dλ/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


def river_proximity(lat, lon, radius=RIVER_DANGER_RADIUS_M):
    """0 = far from all rivers, 1 = at river bank"""
    min_dist = min(haversine(lat, lon, s['lat'], s['lon']) for s in RIVER_STATIONS)
    return max(0.0, 1.0 - min_dist / radius)


ROAD_PENALTIES = {
    'motorway': 0.02, 'motorway_link': 0.03,
    'trunk':    0.05, 'trunk_link':    0.06,
    'primary':  0.10, 'primary_link':  0.12,
    'secondary':0.20, 'secondary_link':0.22,
    'tertiary': 0.30, 'tertiary_link': 0.32,
    'residential': 0.42,
    'unclassified': 0.50,
    'service':  0.55,
    'track':    0.80,
    'path':     0.95,
}

def road_penalty(highway):
    if isinstance(highway, list):
        highway = highway[0]
    return ROAD_PENALTIES.get(str(highway), 0.40)


print('Feature functions defined ✓')

In [ ]:
def build_flood_weights(G, water_pressure, dam_pressure):
    """
    For every edge, compute:
      flood_risk  ∈ [0, 1]   — how flood-prone is this road segment
      flood_cost  ∈ [length, 10×length]  — A* edge weight

    Feature weights (sum = 1.0):
      f_river   0.40  — proximity to Ping / Kuang / Mae Taeng rivers
      f_water   0.25  — live river level vs warning threshold
      f_dam     0.20  — dam fill % (release risk)
      f_road    0.15  — road type (dirt tracks flood worse)
    """
    n_edges = G.number_of_edges()
    print(f'Computing flood weights for {n_edges:,} edges...')

    for u, v, key, data in G.edges(keys=True, data=True):
        u_node = G.nodes[u]
        v_node = G.nodes[v]
        mid_lat = (u_node['y'] + v_node['y']) / 2
        mid_lon = (u_node['x'] + v_node['x']) / 2

        f_river = river_proximity(mid_lat, mid_lon)
        f_road  = road_penalty(data.get('highway', 'residential'))
        f_water = min(water_pressure, 1.5)  # live signal
        f_dam   = min(dam_pressure, 1.1)    # live signal

        flood_risk = (
            0.40 * f_river +
            0.25 * f_water +
            0.20 * f_dam   +
            0.15 * f_road
        )
        flood_risk = min(flood_risk, 1.0)

        base_length = data.get('length', 50.0)
        flood_cost  = base_length * (1.0 + FLOOD_PENALTY_MAX * flood_risk)

        G[u][v][key]['flood_risk'] = round(flood_risk, 4)
        G[u][v][key]['flood_cost'] = round(flood_cost, 2)

    risks = [d['flood_risk'] for _, _, d in G.edges(data=True)]
    print(f'Done — risk stats: min={min(risks):.3f}  mean={np.mean(risks):.3f}  '
          f'p90={np.percentile(risks,90):.3f}  max={max(risks):.3f}')

build_flood_weights(G, WATER_PRESSURE, DAM_PRESSURE)

## 5 · A* Flood-Safe Routing

In [ ]:
def flood_astar(G, origin_latlon, dest_latlon, weight='flood_cost'):
    """Return node list for A* path with haversine heuristic."""
    orig_node = ox.nearest_nodes(G, origin_latlon[1], origin_latlon[0])
    dest_node = ox.nearest_nodes(G, dest_latlon[1],   dest_latlon[0])

    def heuristic(u, v):
        return haversine(G.nodes[u]['y'], G.nodes[u]['x'],
                         G.nodes[v]['y'], G.nodes[v]['x'])

    path = nx.astar_path(G, orig_node, dest_node,
                          heuristic=heuristic, weight=weight)
    return path


def path_stats(G, path):
    pairs = list(zip(path[:-1], path[1:]))
    lengths    = [G[u][v][0].get('length',     0.0) for u, v in pairs]
    flood_costs= [G[u][v][0].get('flood_cost', 0.0) for u, v in pairs]
    risks      = [G[u][v][0].get('flood_risk', 0.0) for u, v in pairs]
    return {
        'nodes':        len(path),
        'distance_km':  round(sum(lengths) / 1000, 2),
        'flood_cost':   round(sum(flood_costs)),
        'avg_risk':     round(np.mean(risks), 3),
        'max_risk':     round(max(risks), 3),
        'risk_pct':     round(np.mean(risks) * 100, 1),
    }


print('A* functions defined ✓')

In [ ]:
results = {}

for route_id, ep in ROUTE_ENDPOINTS.items():
    print(f'\n══ Route {route_id} — {ep["name"]} ══')

    flood_path   = flood_astar(G, ep['origin'], ep['dest'], weight='flood_cost')
    shortest_path = flood_astar(G, ep['origin'], ep['dest'], weight='length')

    fs = path_stats(G, flood_path)
    ss = path_stats(G, shortest_path)

    risk_delta = round((ss['avg_risk'] - fs['avg_risk']) / max(ss['avg_risk'], 0.001) * 100, 1)
    dist_delta = round((fs['distance_km'] - ss['distance_km']) / max(ss['distance_km'], 0.1) * 100, 1)

    print(f'  Flood-safe path:  {fs["distance_km"]} km  avg_risk={fs["avg_risk"]}  max_risk={fs["max_risk"]}')
    print(f'  Shortest path:    {ss["distance_km"]} km  avg_risk={ss["avg_risk"]}  max_risk={ss["max_risk"]}')
    print(f'  → Risk reduced by {risk_delta}%  |  Distance +{dist_delta}%')

    # Use ML score from server if available, otherwise use graph avg
    ml_risk = live_data['route_risk'].get(route_id, {}).get('risk', fs['risk_pct'])

    results[route_id] = {
        'flood':   (flood_path,    fs),
        'shortest':(shortest_path, ss),
        'ml_risk': ml_risk,
        'color':   ep['color'],
        'name':    ep['name'],
    }

## 6 · Visualize on Folium Map

In [ ]:
def path_latlon(G, path):
    return [(G.nodes[n]['y'], G.nodes[n]['x']) for n in path]


m = folium.Map(
    location=[19.908, 99.832],

    tiles='CartoDB dark_matter',
    zoom_start=11,
)

# ── High-risk road overlay (flood heat)
high_risk = [(u, v, d) for u, v, d in G.edges(data=True) if d.get('flood_risk', 0) > 0.45]
for u, v, d in high_risk[:800]:  # cap for browser performance
    coords = [(G.nodes[u]['y'], G.nodes[u]['x']),
              (G.nodes[v]['y'], G.nodes[v]['x'])]
    opacity = min(0.7, d['flood_risk'])
    folium.PolyLine(coords, color='#ef4444', weight=2,
                    opacity=opacity).add_to(m)

# ── Routes
for route_id, data in results.items():
    flood_path,    fs = data['flood']
    shortest_path, ss = data['shortest']
    color = data['color']

    # Shortest (dashed reference)
    folium.PolyLine(
        path_latlon(G, shortest_path),
        color=color, weight=2, opacity=0.35, dash_array='6',
        tooltip=f'Route {route_id} Shortest: {ss["distance_km"]} km | risk {ss["risk_pct"]}%'
    ).add_to(m)

    # Flood-safe (solid)
    folium.PolyLine(
        path_latlon(G, flood_path),
        color=color, weight=5, opacity=0.9,
        tooltip=f'Route {route_id} Flood-safe: {fs["distance_km"]} km | risk {fs["risk_pct"]}%'
    ).add_to(m)

    # Start / end markers
    ep = ROUTE_ENDPOINTS[route_id]
    folium.Marker(
        ep['origin'], icon=folium.Icon(color='blue', icon='play', prefix='fa'),
        tooltip=f'Origin {route_id}'
    ).add_to(m)
    folium.Marker(
        ep['dest'], icon=folium.Icon(color='red', icon='flag', prefix='fa'),
        tooltip=f'Destination {route_id}'
    ).add_to(m)

# ── River stations
for s in RIVER_STATIONS:
    wl_match = next((w for w in live_data['water_levels'] if w['id'] == s['id']), None)
    level_str = f"{wl_match['level']:.2f} m" if wl_match and wl_match.get('level') else 'offline'
    folium.CircleMarker(
        [s['lat'], s['lon']], radius=9,
        color='#3b82f6', fill=True, fill_color='#3b82f6', fill_opacity=0.8,
        tooltip=f'{s["id"]} {s["name"]}: {level_str}'
    ).add_to(m)

# ── Legend (HTML overlay)
legend_html = """
<div style="position:fixed;bottom:30px;left:30px;z-index:9999;
            background:#0b1521dd;border:1px solid #1d6fce44;
            padding:12px 16px;border-radius:8px;font:12px/1.8 Sarabun,sans-serif;color:#e2e8f0">
  <b style="color:#60a5fa">FloodNav Route Legend</b><br>
  <span style="color:#22c55e">━━━</span> Route A — ทล.1 เมือง→แม่สาย<br>
  <span style="color:#f59e0b">━━━</span> Route B — ทล.1020 เมือง→เทิง<br>
  <span style="color:#ef4444">━━━</span> Route C — ทล.118 เมือง→เวียงป่าเป้า<br>
  <span style="color:#ef444466">━━━</span> High flood-risk roads<br>
  <span style="opacity:0.4">- - -</span> Shortest path (reference)
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

map_path = str(ROOT / 'data' / 'flood_routing_map.html')
m.save(map_path)
print(f'Map saved → {map_path}')
display(m)

## 7 · Risk Comparison Table

In [ ]:
rows = []
for rid, data in results.items():
    _, fs = data['flood']
    _, ss = data['shortest']
    rows.append({
        'Route':             f'{rid} — {data["name"]}',
        'Flood-safe dist':   f'{fs["distance_km"]} km',
        'Shortest dist':     f'{ss["distance_km"]} km',
        'Risk (flood-safe)': f'{fs["risk_pct"]}%',
        'Risk (shortest)':   f'{ss["risk_pct"]}%',
        'Risk saved':        f'{round((ss["avg_risk"]-fs["avg_risk"])/max(ss["avg_risk"],0.001)*100,1)}%',
        'ML server risk':    f'{data["ml_risk"]}%',
    })

df = pd.DataFrame(rows).set_index('Route')
display(df)

## 8 · Export GeoJSON → FloodNav Frontend

In [ ]:
def path_to_geojson_feature(G, path, properties):
    coords = [[G.nodes[n]['x'], G.nodes[n]['y']] for n in path]  # [lon, lat] GeoJSON order
    return {
        'type': 'Feature',
        'geometry': {'type': 'LineString', 'coordinates': coords},
        'properties': properties,
    }

geojson = {'type': 'FeatureCollection', 'features': []}

for rid, data in results.items():
    flood_path, fs = data['flood']
    geojson['features'].append(path_to_geojson_feature(G, flood_path, {
        'route_id':        rid,
        'name':            data['name'],
        'type':            'flood_safe',
        'distance_km':     fs['distance_km'],
        'avg_flood_risk':  fs['avg_risk'],
        'max_flood_risk':  fs['max_risk'],
        'ml_risk':         data['ml_risk'],
        'color':           data['color'],
    }))

out_path = str(ROOT / 'data' / 'flood_routes.geojson')
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(geojson, f, ensure_ascii=False, indent=2)

print(f'Exported {out_path}')
for feat in geojson['features']:
    p = feat['properties']
    print(f'  Route {p["route_id"]}: {len(feat["geometry"]["coordinates"])} points, '
          f'{p["distance_km"]} km, risk={p["avg_flood_risk"]}')

## 9 · Weight Sensitivity Analysis
ทดสอบว่าถ้าปรับ live signal (น้ำสูงขึ้น, เขื่อนเต็ม) เส้นทางเปลี่ยนไหม

In [ ]:
scenarios = [
    ('ปกติ',        0.3,  0.5),
    ('ฝนปานกลาง',   0.6,  0.6),
    ('ฝนหนัก',      0.9,  0.8),
    ('เขื่อนเต็ม', 0.95, 1.05),
]

import copy

print(f'{"Scenario":<20} {"Route":<4} {"Flood-safe km":<16} {"Avg risk"}')
print('─' * 55)

for scenario_name, wp, dp in scenarios:
    G_copy = copy.deepcopy(G)
    build_flood_weights(G_copy, wp, dp)
    for rid, ep in ROUTE_ENDPOINTS.items():
        try:
            p  = flood_astar(G_copy, ep['origin'], ep['dest'], 'flood_cost')
            st = path_stats(G_copy, p)
            print(f'{scenario_name:<20} {rid:<4} {str(st["distance_km"]) + " km":<16} {st["avg_risk"]}')
        except Exception as e:
            print(f'{scenario_name:<20} {rid:<4} ERROR: {e}')

---
## Phase 2 Preview — GAT (Graph Attention Network)

เมื่อเก็บ historical flood labels ได้แล้ว ให้ GNN เรียนรู้ edge weights แทน hand-tuned formula

```python
# ต้องการ: pip install torch torch-geometric

import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from torch_geometric.data import Data

class FloodGAT(torch.nn.Module):
    """
    Node features per intersection:
      [lat, lon, dist_to_ping, dist_to_kuang, dist_to_maetaeng,
       water_pressure, dam_pressure, rainfall_mm]
    
    Output per node: flood_risk_score ∈ [0, 1]
    """
    def __init__(self, in_channels=8, hidden=32, heads=4):
        super().__init__()
        self.gat1 = GATConv(in_channels, hidden,  heads=heads, dropout=0.3)
        self.gat2 = GATConv(hidden*heads, 1,       heads=1,    concat=False)

    def forward(self, x, edge_index):
        x = F.elu(self.gat1(x, edge_index))
        x = torch.sigmoid(self.gat2(x, edge_index))  # [0, 1] per node
        return x


def build_pyg_graph(G, live_signals):
    node_list = list(G.nodes())
    node_idx  = {n: i for i, n in enumerate(node_list)}

    # Node feature matrix
    feats = []
    for n in node_list:
        nd   = G.nodes[n]
        lat, lon = nd['y'], nd['x']
        dists = [haversine(lat, lon, s['lat'], s['lon']) / 3000 for s in RIVER_STATIONS]
        feats.append([
            (lat - 19.2) / 1.4,        # normalized lat (CR: 19.2–20.5)
            (lon - 99.3) / 1.2,        # normalized lon (CR: 99.3–100.5)
            *[min(d, 1.0) for d in dists],  # 4 river distances
            live_signals['water_pressure'],
            live_signals['dam_pressure'],
        ])
    x = torch.tensor(feats, dtype=torch.float)

    # Edge index [2, E]
    edges = [(node_idx[u], node_idx[v]) for u, v in G.edges()]
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()

    return Data(x=x, edge_index=edge_index), node_idx


# Training (needs labeled data: which edges flooded in historical events)
# model = FloodGAT(in_channels=8)
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
# loss_fn = torch.nn.BCELoss()  # binary: flooded or not
#
# for epoch in range(200):
#     out = model(data.x, data.edge_index)  # node-level risk
#     loss = loss_fn(out[train_mask], y[train_mask])
#     loss.backward(); optimizer.step(); optimizer.zero_grad()
#
# After training: replace flood_risk edge attribute with GNN output
# → feed into same A* → better risk estimates without manual tuning
```

**ข้อมูลที่ต้องเก็บเพื่อ train GNN:**
- ปี 2554, 2561, 2567 น้ำท่วมเชียงใหม่ — ถนนไหนน้ำท่วม → label = 1  
- ช่วงปกติ — label = 0  
- Source: กรมทางหลวง, กรมป้องกัน, ข่าว/social ย้อนหลัง